# Program 24
## Implement a Machine Translation Pipeline using Transformer Architecture
1. Data Preparation
2. Model Architecture (Transformer from scratch - Simplified)
3. Training
4. Inference

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Embedding, LayerNormalization, Dropout, GlobalAverageNormalization
import numpy as np

# ==========================================
# 1. Data Preparation
# ==========================================
# Very small toy dataset for fast demonstration
fra_sentences = ['bonjour', 'comment allez vous', 'je suis content', 'merci beaucoup', 'au revoir']
eng_sentences = ['[START] hello [END]', '[START] how are you [END]', '[START] i am happy [END]', '[START] thank you very much [END]', '[START] goodbye [END]']

# Tokenizers
src_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
src_tokenizer.fit_on_texts(fra_sentences)
src_vocab_size = len(src_tokenizer.word_index) + 1
src_seq = src_tokenizer.texts_to_sequences(fra_sentences)
src_seq = tf.keras.preprocessing.sequence.pad_sequences(src_seq, padding='post')

tar_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
tar_tokenizer.fit_on_texts(eng_sentences)
tar_vocab_size = len(tar_tokenizer.word_index) + 1
tar_seq = tar_tokenizer.texts_to_sequences(eng_sentences)
tar_seq = tf.keras.preprocessing.sequence.pad_sequences(tar_seq, padding='post')

# Setup Input and Target
X = src_seq
decoder_input = tar_seq[:, :-1]
y = tar_seq[:, 1:]

max_len = X.shape[1]

# ==========================================
# 2. Model Architecture (Transformer)
# ==========================================
def positional_encoding(length, depth):
    depth = depth / 2
    positions = np.arange(length)[:, np.newaxis] 
    depths = np.arange(depth)[np.newaxis, :] / depth
    angle_rates = 1 / (10000**depths)
    angle_rads = positions * angle_rates
    pos_encoding = np.concatenate(
        [np.sin(angle_rads), np.cos(angle_rads)], axis=-1)
    return tf.cast(pos_encoding, dtype=tf.float32)

class PositionalEmbedding(Layer):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = Embedding(vocab_size, d_model, mask_zero=True)
        self.pos_encoding = positional_encoding(length=2048, depth=d_model)

    def call(self, x):
        length = tf.shape(x)[1]
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = x + self.pos_encoding[tf.newaxis, :length, :]
        return x

# Building a simple Encoder-Decoder using functional API for brevity
d_model = 32
num_heads = 2

# Encoder
encoder_inputs = tf.keras.Input(shape=(None,), dtype='int64')
x_enc = PositionalEmbedding(src_vocab_size, d_model)(encoder_inputs)
enc_out = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(x_enc, x_enc)
enc_out = LayerNormalization(epsilon=1e-6)(x_enc + enc_out)

# Decoder
decoder_inputs = tf.keras.Input(shape=(None,), dtype='int64')
x_dec = PositionalEmbedding(tar_vocab_size, d_model)(decoder_inputs)

# Masked attention (prevent looking ahead)
dec_out1 = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(x_dec, x_dec, use_causal_mask=True)
dec_out1 = LayerNormalization(epsilon=1e-6)(x_dec + dec_out1)

# Cross attention
dec_out2 = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(dec_out1, enc_out)
dec_out2 = LayerNormalization(epsilon=1e-6)(dec_out1 + dec_out2)

outputs = Dense(tar_vocab_size, activation='softmax')(dec_out2)

transformer = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs], outputs=outputs)
transformer.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# ==========================================
# 3. Training
# ==========================================
print("Training Transformer Model...")
history = transformer.fit(
    [X, decoder_input], 
    y, 
    epochs=100, 
    verbose=0
)
print("Training Complete. Final loss:", history.history['loss'][-1])

# ==========================================
# 4. Inference
# ==========================================
def translate(sentence):
    encoder_in = src_tokenizer.texts_to_sequences([sentence])
    encoder_in = tf.keras.preprocessing.sequence.pad_sequences(encoder_in, maxlen=X.shape[1], padding='post')
    
    decoder_in = [tar_tokenizer.word_index['[start]']]
    
    for i in range(10):
        dec_in_pad = tf.keras.preprocessing.sequence.pad_sequences([decoder_in], maxlen=decoder_input.shape[1], padding='post')
        predictions = transformer.predict([encoder_in, dec_in_pad], verbose=0)
        predicted_id = np.argmax(predictions[0, i, :])
        
        if predicted_id == tar_tokenizer.word_index.get('[end]', -1):
            break
            
        decoder_in.append(predicted_id)
        
    # Convert IDs to words
    inv_tar_vocab = {v: k for k, v in tar_tokenizer.word_index.items()}
    translated_words = [inv_tar_vocab.get(i, '') for i in decoder_in[1:]]
    return ' '.join(translated_words)

test_sentence = "bonjour"
print(f"\nInput: {test_sentence}")
print(f"Translation: {translate(test_sentence)}")

test_sentence_2 = "merci beaucoup"
print(f"Input: {test_sentence_2}")
print(f"Translation: {translate(test_sentence_2)}")